# 01 · Data collection and cleaning

**Project:** Does ESG risk pay off? ESG risk ratings and stock performance of S&P 500 companies (2019–2025)

**Goal of this notebook:** load the ESG risk ratings, clean them, collect daily stock prices with Python and build a clean sample of companies with both ESG scores and complete price data.

**Sources**
- ESG risk ratings: *S&P 500 ESG Risk Ratings* (Kaggle), based on Sustainalytics scores (2023). A **lower** score means **lower** unmanaged ESG risk.
- Stock prices: Yahoo Finance, collected with the `yfinance` library.

**Outputs:** `data/esg_clean.csv`, `data/prices_raw.csv`

## 1. Load the ESG dataset
503 companies and 15 columns. First issues detected with `info()`:
- 73 companies have no ESG score.
- `Full Time Employees` and `ESG Risk Percentile` are stored as text.
- `Controversy Score` has additional missing values.

In [10]:
import pandas as pd
esg = pd.read_csv("../data/SP 500 ESG Risk Ratings.csv")
esg.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Symbol                  503 non-null    str    
 1   Name                    503 non-null    str    
 2   Address                 502 non-null    str    
 3   Sector                  502 non-null    str    
 4   Industry                502 non-null    str    
 5   Full Time Employees     498 non-null    str    
 6   Description             502 non-null    str    
 7   Total ESG Risk score    430 non-null    float64
 8   Environment Risk Score  430 non-null    float64
 9   Governance Risk Score   430 non-null    float64
 10  Social Risk Score       430 non-null    float64
 11  Controversy Level       430 non-null    str    
 12  Controversy Score       403 non-null    float64
 13  ESG Risk Percentile     430 non-null    str    
 14  ESG Risk Level          430 non-null    str    
dtype

## 2. Select relevant columns
`Address` and `Description` are dropped as they are not relevant for the analysis.

In [11]:
columnas = [
    "Symbol", "Name", "Sector", "Industry", "Full Time Employees",
    "Total ESG Risk score", "Environment Risk Score",
    "Social Risk Score", "Governance Risk Score",
    "Controversy Level", "Controversy Score",
    "ESG Risk Percentile", "ESG Risk Level"
]
esg = esg[columnas].copy()

## 3. Rename columns
Short, lowercase names without spaces make the code easier to write and read.

In [12]:
esg.columns = [
    "ticker", "company_name", "sector", "industry", "employees",
    "esg_total", "esg_environmental", "esg_social", "esg_governance",
    "controversy_level", "controversy_score",
    "esg_percentile", "esg_level"
]
esg.head()

,ticker,company_name,sector,industry,employees,esg_total,esg_environmental,esg_social,esg_governance,controversy_level,controversy_score,esg_percentile,esg_level
0,ENPH,"Enphase Energy, Inc.",Technology,Solar,"3,157",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EMN,Eastman Chemical Company,Basic Materials,Specialty Chemicals,"14,000",25.3,12.8,5.8,6.6,Moderate Controversy Level,2.0,50th percentile,Medium
2,DPZ,Domino's Pizza Inc.,Consumer Cyclical,Restaurants,"6,500",29.2,10.6,12.2,6.3,Moderate Controversy Level,2.0,66th percentile,Medium
3,DAY,"Dayforce, Inc.",Technology,Software - Application,"9,084",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DVA,Davita Inc.,Healthcare,Medical Care Facilities,"70,000",22.6,0.1,14.1,8.4,Moderate Controversy Level,2.0,38th percentile,Medium


## 4. Convert text columns to numeric
`employees` contains thousands separators and `esg_percentile` is stored as text ("50th percentile"), so both are converted to numbers. `float` is used because both columns contain missing values.

In [13]:
# Remove thousands separators and convert to numeric
esg["employees"] = esg["employees"].str.replace(",", "").astype(float)

# Extract the number from text like "50th percentile"
esg["esg_percentile"] = esg["esg_percentile"].str.extract(r"(\d+)").astype(float)

esg.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ticker             503 non-null    str    
 1   company_name       503 non-null    str    
 2   sector             502 non-null    str    
 3   industry           502 non-null    str    
 4   employees          498 non-null    float64
 5   esg_total          430 non-null    float64
 6   esg_environmental  430 non-null    float64
 7   esg_social         430 non-null    float64
 8   esg_governance     430 non-null    float64
 9   controversy_level  430 non-null    str    
 10  controversy_score  403 non-null    float64
 11  esg_percentile     430 non-null    float64
 12  esg_level          430 non-null    str    
dtypes: float64(7), str(6)
memory usage: 51.2 KB


## 5. Handle missing ESG scores
Before removing companies without an ESG score, check whether they are concentrated in any sector, which would bias the analysis.

In [14]:
# Companies without ESG score
missing_esg = esg[esg["esg_total"].isna()]
print(f"Companies without ESG score: {len(missing_esg)}")

# Which sectors are they in?
missing_esg["sector"].value_counts()

Companies without ESG score: 73


sector
Technology                15
Industrials               13
Healthcare                12
Communication Services     8
Consumer Cyclical          6
Utilities                  4
Real Estate                3
Financial Services         3
Basic Materials            3
Consumer Defensive         3
Energy                     2
Name: count, dtype: int64

73 companies (14.5%) have no ESG score. They are spread across all sectors, so removing them does not exclude any sector from the analysis. The result is stored in a new table, `esg_clean`, keeping the original intact.

Missing `controversy_score` values are **not** removed, since those companies do have an ESG score.

In [15]:
# Keep only companies with an ESG score
esg_clean = esg.dropna(subset=["esg_total"]).copy()
print(f"Companies before: {len(esg)} | after: {len(esg_clean)}")

Companies before: 503 | after: 430


## 6. Collect stock prices with `yfinance`
Test the download with a single ticker first. The period 2019–2025 includes two very different market stress episodes: the COVID crash (2020) and the rate-hike sell-off (2022). `Close` prices are adjusted for dividends and splits.

In [20]:
import yfinance as yf

# Test download with a single ticker
test = yf.download("AAPL", start="2019-01-01", end="2025-12-31")
test.head()


[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
[*********************100%***********************]  1 of 1 completed

Note: you may need to restart the kernel to use updated packages.


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2019-01-02,37.436913,37.657383,36.562151,36.718613,148158800
2019-01-03,33.707916,34.544747,33.662874,34.132257,365248800
2019-01-04,35.146893,35.215643,34.089596,34.262650,234428400
2019-01-07,35.068668,35.282027,34.587432,35.251208,219111200
2019-01-08,35.737175,35.990834,35.208527,35.455070,164101200


Download daily closing prices for all companies. Yahoo Finance uses `-` instead of `.` in tickers (e.g. `BRK.B` → `BRK-B`).

In [21]:
# Yahoo Finance uses "-" instead of "." in tickers (e.g. BRK.B -> BRK-B)
tickers = esg_clean["ticker"].str.replace(".", "-", regex=False).tolist()

# Download closing prices for all tickers
prices = yf.download(tickers, start="2019-01-01", end="2025-12-31")["Close"]
print(prices.shape)

[*****                 10%                       ]  45 of 430 completed$MRO: possibly delisted; no timezone found
[*****                 11%                       ]  49 of 430 completed$CMA: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
[******                12%                       ]  50 of 430 completed$MMC: possibly delisted; no timezone found
[********              17%                       ]  73 of 430 completed$EA: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
[*********             19%                       ]  81 of 430 completed$AVB: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
[***********           23%                       ]  97 of 430 completed$DFS: possibly delisted; no timezone found
[****************      34%                       ]  148 of 430 completed$FI: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
[******************    37%                       ]  159 of 430 completed$QRVO: 

(1759, 430)


## 7. Handle failed downloads
18 tickers failed on the first download. Failed tickers are retried once to rule out temporary download errors.

In [22]:
# Tickers with no data at all
failed = prices.columns[prices.isna().all()].tolist()
print(f"Failed tickers: {len(failed)}")

# Retry download for failed tickers only
retry = yf.download(failed, start="2019-01-01", end="2025-12-31")["Close"]
prices.update(retry)

# Check which ones still have no data
still_failed = prices.columns[prices.isna().all()].tolist()
print(f"Still failing after retry: {len(still_failed)}")
print(still_failed)

Failed tickers: 18


$K: possibly delisted; no price data found  (1d 2019-01-01 -> 2025-12-31)
[                       0%                       ]$IPG: possibly delisted; no price data found  (1d 2019-01-01 -> 2025-12-31)
[*****                 11%                       ]  2 of 18 completed$CMA: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
$JNPR: possibly delisted; no timezone found
[********              17%                       ]  3 of 18 completed$EQR: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
$EA: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
[*************         28%                       ]  5 of 18 completed$QRVO: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
[****************      33%                       ]  6 of 18 completed$BK: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
[****************      33%                       ]  6 of 18 completed$AVB: Data doesn't exist for startDate = 1

Still failing after retry: 18
['ANSS', 'AVB', 'BK', 'CMA', 'CTRA', 'DFS', 'EA', 'EQR', 'FI', 'HES', 'HOLX', 'IPG', 'JNPR', 'K', 'MMC', 'MRO', 'QRVO', 'WBA']


The 18 tickers still fail after the retry. Most correspond to companies acquired, merged or renamed since the ESG data was collected (2023). They are removed from **both** tables so that ESG and price data cover exactly the same companies.

**Limitation:** this introduces survivorship bias, since only companies with full price history are analysed.

In [23]:
# Drop tickers without price data from both tables
prices = prices.drop(columns=still_failed)
esg_clean = esg_clean[~esg_clean["ticker"].isin(still_failed)].copy()

print(f"Companies with ESG and prices: {len(esg_clean)}")
print(f"Price table shape: {prices.shape}")

Companies with ESG and prices: 412
Price table shape: (1759, 412)


Check: downloading one failing ticker on its own (BK) also fails, so the issue is not a temporary block from Yahoo Finance.

In [25]:
yf.download("BK", start="2019-01-01", end="2025-12-31")["Close"].head()

$BK: Data doesn't exist for startDate = 1546318800, endDate = 1767157200
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['BK']: Data doesn't exist for startDate = 1546318800, endDate = 1767157200


Ticker,BK


## 8. Check price completeness
Share of trading days with no price, per company.

In [26]:
# Share of missing days per company (top 10)
missing_share = prices.isna().mean().sort_values(ascending=False)
missing_share.head(10)

Ticker
PARA    0.303013
ABBV    0.000000
ABT     0.000000
ACGL    0.000000
ACN     0.000000
ADBE    0.000000
ADI     0.000000
ADM     0.000000
ADP     0.000000
ADSK    0.000000
dtype: float64

Only PARA (Paramount) has missing prices (30% of trading days). Its first available price is in February 2021.

In [27]:
print(prices["PARA"].first_valid_index())
print(prices["PARA"].last_valid_index())

2021-02-12 00:00:00
2025-12-30 00:00:00


PARA is removed so that all companies are compared over the same period. Final sample: **411 companies** with an ESG score and complete prices.

In [28]:
# Drop PARA: 30% of the period has no price data
prices = prices.drop(columns=["PARA"])
esg_clean = esg_clean[esg_clean["ticker"] != "PARA"].copy()

print(f"Final sample: {len(esg_clean)} companies")

prices.to_csv("../data/prices_raw.csv")
esg_clean.to_csv("../data/esg_clean.csv", index=False)

Final sample: 411 companies


## Summary
| Step | Companies |
|---|---|
| Raw ESG dataset | 503 |
| With ESG score | 430 |
| With price data | 412 |
| With complete price data (final sample) | **411** |

Next notebook: `02_metrics.ipynb`, where performance and risk metrics are computed for each company.